# 03b - RM-b: frozen encoder + classification head

Encoder IndoBERT dibekukan dan hanya classification head yang dilatih. Embedding
mean-pool diekstrak sekali lalu dipakai ulang oleh seluruh konfigurasi head,
sehingga satu epoch hanya berupa perkalian matriks kecil.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [1]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-14 16:48:57,271 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device        : cuda
encoder       : indobenchmark/indobert-base-p2
keluaran      : /workspace/indobert-with-rac/outputs/baseline
train/val/test: [6588, 1402, 1405]


## 1. Ekstraksi fitur beku

In [2]:
features = runner.features

print(f"dari cache      : {features.from_cache}")
print(f"dimensi         : {features.hidden_dim}")
print(f"waktu ekstraksi : {features.extract_time_s:.2f} s")
print(f"peak GPU memory : {features.extract_peak_mem_mb:.0f} MB")
for split in ("train", "val", "test"):
    embeddings, labels = features[split]
    print(f"  {split:5s}: {embeddings.shape} | label {labels.shape}")

2026-09-14 16:48:58,460 | INFO     | src.services.features | Ekstraksi fitur beku dengan encoder indobenchmark/indobert-base-p2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 60862.37it/s]

2026-09-14 16:48:59,777 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3


2026-09-14 16:49:04,975 | INFO     | src.services.features |   train: (6588, 768)
2026-09-14 16:49:06,046 | INFO     | src.services.features |   val: (1402, 768)
2026-09-14 16:49:07,124 | INFO     | src.services.features |   test: (1405, 768)
2026-09-14 16:49:07,158 | INFO     | src.services.features | Fitur beku disimpan ke /workspace/indobert-with-rac/outputs/baseline/features/indobenchmark__indobert-base-p2
dari cache      : False
dimensi         : 768
waktu ekstraksi : 7.30 s
peak GPU memory : 530 MB
  train: (6588, 768) | label (6588,)
  val  : (1402, 768) | label (1402,)
  test : (1405, 768) | label (1405,)


Ekstraksi berjalan sekali per encoder lalu di-cache ke
`outputs/baseline/features/<nama_encoder>/`. Biaya ini tetap dihitung sebagai
bagian waktu latih RM-b agar perbandingannya dengan RM-a jujur, walau
diamortisasi ke seluruh konfigurasi head.

## 2. Konfigurasi

In [3]:
from src.models.schemas import RMBConfig

config = RMBConfig()
print(config.model_dump())

{'head_arch': 'linear', 'hidden_dim': 256, 'epochs': 5, 'lr': 0.0002, 'dropout': 0.1, 'weight_decay': 0.01, 'batch': 32, 'seed': 42}


Default mengikuti Peters dkk. (2019) untuk transfer berbasis fitur: head
diinisialisasi acak sehingga butuh learning rate lebih tinggi daripada RM-a.

## 3. Jalankan

In [4]:
row = runner.run(
    "rmb",
    config.model_dump(),
    note="baseline head linear di atas fitur beku",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro     : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi      : {row['val_f1_judi']:.4f}")
print(f"  epoch terbaik    : {row['best_epoch']} dari {row['epochs']}")
print(f"  waktu latih head : {row['head_train_time_s']:.2f} s")
print(f"  + ekstraksi      : {row['extract_time_s']:.2f} s")
print(f"  total waktu latih: {row['train_time_s']:.2f} s")
print(f"  trainable params : {row['trainable_params']:,}")

2026-09-14 16:49:07,167 | INFO     | src.services.campaign | [rmb] RUN #1 {'head_arch': 'linear', 'hidden_dim': 256, 'epochs': 5, 'lr': 0.0002, 'dropout': 0.1, 'weight_decay': 0.01, 'batch': 32, 'seed': 42}
2026-09-14 16:49:08,055 | INFO     | src.services.run_log | Juara baru untuk rmb: val F1-macro 0.9125 (sebelumnya -1.0000)
2026-09-14 16:49:08,189 | INFO     | src.services.campaign | [rmb] RUN #1 val F1-macro 0.9125 (epoch terbaik 5)
run #1
  val F1-macro     : 0.9125
  val F1 judi      : 0.8597
  epoch terbaik    : 5 dari 5
  waktu latih head : 0.72 s
  + ekstraksi      : 7.30 s
  total waktu latih: 8.02 s
  trainable params : 1,538


## 4. Bandingkan head linear dan MLP

In [5]:
row_mlp = runner.run(
    "rmb",
    {"head_arch": "mlp", "hidden_dim": 256},
    note="uji apakah satu hidden layer menutup selisih terhadap RM-a",
)

print(f"linear: F1-macro {row['val_f1_macro']:.4f} | "
      f"{row['trainable_params']:,} params")
print(f"mlp   : F1-macro {row_mlp['val_f1_macro']:.4f} | "
      f"{row_mlp['trainable_params']:,} params")
print(f"selisih: {(row_mlp['val_f1_macro'] - row['val_f1_macro']) * 100:+.2f} pp")

2026-09-14 16:49:08,197 | INFO     | src.services.run_log | Riwayat runs_rmb.csv dimuat: 1 run
2026-09-14 16:49:08,197 | INFO     | src.services.campaign | [rmb] RUN #2 {'head_arch': 'mlp', 'hidden_dim': 256, 'epochs': 5, 'lr': 0.0002, 'dropout': 0.1, 'weight_decay': 0.01, 'batch': 32, 'seed': 42}
2026-09-14 16:49:09,094 | INFO     | src.services.run_log | Juara baru untuk rmb: val F1-macro 0.9400 (sebelumnya 0.9125)
2026-09-14 16:49:09,229 | INFO     | src.services.campaign | [rmb] RUN #2 val F1-macro 0.9400 (epoch terbaik 5)
linear: F1-macro 0.9125 | 1,538 params
mlp   : F1-macro 0.9400 | 197,378 params
selisih: +2.75 pp


## Ringkasan

Perbandingan trainable parameter terhadap RM-a adalah inti klaim efisiensi
skenario ini. Pencarian kapasitas head yang optimal dilakukan di
`04_tuning_campaign.ipynb`.

Lanjut ke `03c_rmc_rac.ipynb`.